In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from enum import Enum
from numba.typed import Dict
from numba import types
from numba import njit, float32, int64, float64,types
from numba.typed import Dict
import joblib

In [2]:
#################################
# 1. Load address
#################################
Base_dir = "C:/Users/user/Desktop/IDS_masters/9) Car-Hacking Dataset"

attack_dir = {
    "Dos" : 1,
    "Fuzzing" :2 ,
    "Spoofing" : 4
}


### 공격 파일 수집 ###
attack_files = []

for folder, attack_id in attack_dir.items():
    folder_path = os.path.join(Base_dir,folder)

    for fname in os.listdir(folder_path):
        if fname.endswith(".csv"):
            attack_files.append({
                "path" : os.path.join(folder_path, fname),
                "attack_id": attack_id
            })

In [3]:
def build_transition_ref_map(can_ids):
    """
    정상 패킷들의 ID 시퀀스를 분석하여 전이 확률 맵을 생성합니다.
    """
    # 1. 빈도수 카운트를 위한 임시 딕셔너리
    counts = {}
    total_transitions = {}

    prev_id = -1
    for cid in can_ids:
        cid = int(cid)
        if prev_id != -1:
            key = (prev_id << 32) | cid
            counts[key] = counts.get(key, 0) + 1
            total_transitions[prev_id] = total_transitions.get(prev_id, 0) + 1
        prev_id = cid

    # 2. Numba 호환 Dict 생성 (Key: int64, Value: float64)
    ref_map = Dict.empty(key_type=types.int64, value_type=types.float64)

    # 3. 빈도수를 확률로 변환
    for key, count in counts.items():
        prev_id_part = key >> 32
        # P(현재ID | 이전ID) 계산
        prob = count / total_transitions[prev_id_part]
        ref_map[key] = prob

    return ref_map

In [4]:
#################################
# 2. Visualization Mirgu Dataset
#################################

# [ADD] payload 8바이트 리스트로 만드는 함수 (너가 쓰던 스타일)
def parse_payload(row):
    # row에는 b0~b7 컬럼이 있고, 이미 0패딩되어 있음
    return [int(row[f"b{i}"]) for i in range(8)]

def process_csv_file(path, attack_id):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            if len(parts) < 4:
                continue

            ts_str, canid_raw, dlc_str = parts[0], parts[1], parts[2]
            label = parts[-1].strip()         # [MINOR] strip
            data_tokens = parts[3:-1]

            # dlc/ts 파싱
            try:
                ts = float(ts_str)
                dlc = int(dlc_str)
            except:
                continue

            # payload bytes: DLC 만큼만 읽고, 8바이트로 0 패딩
            payload = []
            for i in range(min(dlc, len(data_tokens), 8)):
                tok = data_tokens[i].strip()
                if tok == "" or tok.lower() == "nan":
                    payload.append(0)
                else:
                    try:
                        payload.append(int(tok, 16))
                    except:
                        payload.append(0)

            payload += [0] * (8 - len(payload))
            payload = payload[:8]

            rows.append([ts, canid_raw, dlc, *payload, label])

    df = pd.DataFrame(
        rows,
        columns=["timestamp", "CAN_ID", "DLC"] + [f"b{i}" for i in range(8)] + ["Label"]
    )

    # CAN_ID int 변환
    df["int_CAN_ID"] = df["CAN_ID"].apply(lambda x: int(str(x).strip(), 16)).astype(np.int64)


    # Payloads 컬럼 추가 
    df["Payloads"] = df.apply(parse_payload, axis=1).tolist()

    # 라벨링
    df["Labeling"] = df["Label"].map({"T": attack_id, "R": 0}).fillna(0).astype(int)

    df = df[["timestamp","int_CAN_ID","Payloads","Labeling"]]

    return df


In [5]:
# %%
@njit
def popcount64(x):
    # x: uint8 -> 0~255
    c = 0
    v = x
    while v:
        v &= v - np.uint64(1)
        c += 1
    return c

@njit
def pack_payload_u64(row):
    v = np.uint64(0)
    for i in range(8):
        v |= np.uint64(row[i]) << (i * 8)
    return v

@njit(fastmath=True)
def calculate_features_numba(timestamps, can_ids, state_codes, payloads, transition_ref_map):
    n = len(timestamps)
    # 기존 8개 -> payload Z-score 1개 추가해서 9개
    features = np.zeros((n, 9), dtype=np.float32)
    
    last_time_map   = Dict.empty(key_type=types.int64, value_type=types.float64)   # 이전 패킷 시간
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)   # 이전 ID Payload
    last_id_map     = Dict.empty(key_type=types.int64, value_type=types.int64)     # 윈도우 내 id 빈도수
    ent_history_map = Dict.empty(key_type=types.int64, value_type=types.float64[:])# 엔트로피 히스토리
    # 새로 추가: per-ID, per-byte payload 통계 히스토리
    # [byte0_sum..byte7_sum, byte0_sumsq..byte7_sumsq, count] -> 길이 17
    payload_hist_map = Dict.empty(key_type=types.int64, value_type=types.float64[:])

    prev_state = np.int64(-1)
    prev_global_time = timestamps[0]
    eps = 1e-9

    # --- Markov 관련 변수 ---
    prev_id = np.int64(-1)
    min_prob = 1e-5  # 기준 행렬에 없는 전이에 부여할 최소 확률

    prev_global_time = timestamps[0]

    for i in range(n):
        # 윈도우마다 ID 빈도수 초기화
        if (i % 64) == 0:
            last_id_map.clear()

        ts = timestamps[i]
        cid = can_ids[i]
        sc = state_codes[i]

        if np.isnan(ts):
            ts = prev_global_time
        else:
            prev_global_time = ts

        row = payloads[i]

        # 1. [Index 0] ID IAT
        if cid in last_time_map:
            id_iat = max(0.0, ts - last_time_map[cid])
        else:
            id_iat = 0.0
        features[i, 0] = float32(np.log1p(id_iat * 1000.0) / 7.0)
        last_time_map[cid] = ts  # 다음 계산을 위해 업데이트

        # --- 페이로드 관련 공통 준비 (Entropy용) ---
        counts = np.zeros(256, dtype=np.int32)
        for b_idx in range(8):
            val = row[b_idx]
            counts[val] += 1

        # 2. [Index 1] : ID가 0x000인지 여부(Dos 구분)
        is_zero_id = 1.0 if cid == 0 else 0.0
        features[i, 1] = float32(is_zero_id)

        # 3. [Index 2] Entropy
        ent = 0.0
        for c in counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)
        features[i, 2] = float32(ent / 2.1)

        # 4. ID Hamming 관련
        cur_bytes = pack_payload_u64(row)
        if cid in last_payload_map:
            diff = cur_bytes ^ last_payload_map[cid]
            id_ham = popcount64(diff)
        else:
            id_ham = 0
        last_payload_map[cid] = cur_bytes

        # 4. [Index 3] complexity = entropy * hamming
        compelxity = ent * id_ham
        if compelxity == 0:
            features[i, 3] = float32(0.0)
        else:
            features[i, 3] = float32(np.log1p(compelxity))

        # 5. [Index 4] hamming rate = hamming / IAT
        ham_rate = id_ham / (id_iat + eps)
        if ham_rate == 0:
            features[i, 4] = float32(0.0)
        else:
            features[i, 4] = float32(np.log1p(ham_rate))

        # (선택) DoS가 아닌 ID에서는 payload 관련 피쳐를 좀 더 강조하고 싶으면:
        # if cid != 0:
        #     features[i, 3] *= 1.5
        #     features[i, 4] *= 1.5

        # 6. [Index 5] Frequency (윈도우 내 ID 등장 횟수)
        if cid in last_id_map:
            cnt = last_id_map[cid] + 1
        else:
            cnt = 1
        last_id_map[cid] = int64(cnt)
        features[i, 5] = float32(cnt / 64.0)

        # 7. [Index 6] Markov Score (전달받은 transition_ref_map 사용)
        if prev_state != -1:
            transition_key = (prev_state << 32) | sc
            if transition_key in transition_ref_map:
                prob = transition_ref_map[transition_key]
            else:
                prob = min_prob
            features[i, 6] = float32(-np.log(prob) / 11.5)
        else:
            features[i, 6] = float32(0.0)
        prev_state = sc

        # ---------------------------------------------------------
        #  엔트로피 히스토리 기반 Z-score (기존 Index 7)
        # ---------------------------------------------------------
        if cid not in ent_history_map:
            # [합계, 제곱합, 샘플수] 초기화
            ent_history_map[cid] = np.array([0.0, 0.0, 0.0], dtype=np.float64)

        hist = ent_history_map[cid]
        n_prev = hist[2]  # 지금까지 쌓인 개수

        final_ent_val = 0.0
        if n_prev > 30:
            avg_ent = hist[0] / n_prev
            var_ent = (hist[1] / n_prev) - (avg_ent ** 2)
            std_ent = np.sqrt(np.maximum(var_ent, 0.0))
            z_ent = (ent - avg_ent) / (std_ent + eps)
            # Clipping (-7.0 ~ 7.0)
            final_ent_val = np.maximum(np.minimum(z_ent, 7.0), -7.0)

        features[i, 7] = np.float32(final_ent_val)

        # 엔트로피 히스토리 업데이트
        hist[0] += ent          # 합계 누적
        hist[1] += ent ** 2     # 제곱합 누적
        hist[2] += 1.0          # 카운트 증가

        # ---------------------------------------------------------
        #  새로 추가: per-ID, per-byte payload Z-score (Index 8)
        # ---------------------------------------------------------
        if cid not in payload_hist_map:
            # [byte0_sum..byte7_sum, byte0_sumsq..byte7_sumsq, count]
            payload_hist_map[cid] = np.zeros(17, dtype=np.float64)

        phist = payload_hist_map[cid]
        count_p = phist[16]

        payload_z = 0.0
        if count_p > 30:
            max_abs_z = 0.0
            for b_idx in range(8):
                s = phist[b_idx]
                sq = phist[8 + b_idx]
                mu = s / count_p
                var = (sq / count_p) - mu * mu
                std = np.sqrt(np.maximum(var, 0.0)) + eps

                z = (row[b_idx] - mu) / std
                abs_z = np.abs(z)
                if abs_z > max_abs_z:
                    max_abs_z = abs_z

            # 0 ~ 7 사이로 클리핑 (양수로만 사용)
            if max_abs_z > 7.0:
                max_abs_z = 7.0
            payload_z = max_abs_z

        features[i, 8] = np.float32(payload_z)

        # payload 히스토리 업데이트
        for b_idx in range(8):
            val = row[b_idx]
            phist[b_idx]     += val
            phist[8 + b_idx] += val * val
        phist[16] += 1.0

    return features


In [6]:
# 기존 Make_feature를 이 형태로 덮어씌우세요
def Make_feature(path, attack_id,timestamps, can_ids, state_codes, payloads, labels, transition_ref_map):
    # 1. 파일 읽기
    df = process_csv_file(path, attack_id)

    # 2. 넘파이 배열로 변환
    timestamps = df["timestamp"].to_numpy(np.float32)
    can_ids = df["int_CAN_ID"].to_numpy(np.int64)
    payloads = np.array(df["Payloads"].tolist(), dtype=np.uint8)
    labels = df["Labeling"].to_numpy(np.int64)

    # 3. 피처 계산 (Numba 함수 호출)
    # [수정] 인자로 받은 state_codes를 추가로 넘겨줍니다.
    feature9 = calculate_features_numba(
        timestamps, 
        can_ids, 
        state_codes, 
        payloads, 
        transition_ref_map
    )

    print(f"Feature 추출 완료: {feature9.shape}")
    return feature9, labels

In [7]:
# ==========================================
# 5. Slide Window and Label
# ==========================================
def Sliding_Window_and_Labeling(feature, label, win_size=128, stride=64):
    windows = []
    labels = []
    n = feature.shape[0]
    for start in range(0, n-win_size+1 , stride):
        end = start + win_size
        windows.append(feature[start:end])
        labels.append(label[start:end])


    return (
        np.stack(windows, axis=0).astype(np.float32),
        np.stack(labels, axis=0).astype(np.int64)
    )

In [8]:
all_x = []
all_y = []

map_path = "C:/Users/user/Desktop/IDS_masters/dataset/"

def py_to_numba_transition_map(py_map):
    nb_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    for k, v in py_map.items():
        nb_map[np.int64(k)] = float(v)
    return nb_map

id_to_state = joblib.load(os.path.join(map_path, "id_to_state.pkl"))
global_transition_ref_map = joblib.load(os.path.join(map_path, "markov_ref_map.pkl"))
transition_ref_map_nb = py_to_numba_transition_map(global_transition_ref_map)



print("🚀 Extracting features for Evaluation Dataset...")
for item in attack_files:
    # 1. 파일 읽기
    df_temp = process_csv_file(item["path"], item["attack_id"])
    
    timestamps = df_temp["timestamp"].to_numpy(np.float32)
    can_ids = df_temp["int_CAN_ID"].to_numpy(np.int64)
    payload_array = np.array(df_temp["Payloads"].tolist(), dtype=np.uint8)
    labels = df_temp["Labeling"].to_numpy(np.int64)

    # 2. 훈련 시 기준을 적용해 ID를 State로 변환 (모르는 ID는 9:Event)
    state_sequences = np.array([id_to_state.get(cid, 9) for cid in can_ids], dtype=np.int32)

    # 3. 피처 계산 (훈련 때의 맵 주입)
    # [주의] calculate_features_numba 함수가 state_codes를 받도록 수정되어 있어야 합니다!
     # 3. 피처 계산 (훈련 때의 Markov 맵 사용)
    feature9, labels = Make_feature(
        item["path"],
        item["attack_id"],
        timestamps,
        can_ids,
        state_sequences,
        payload_array,
        labels,
        transition_ref_map_nb
    )
    # 4. 윈도우 슬라이싱
    windows, y = Sliding_Window_and_Labeling(feature9, labels)
    all_x.append(windows)
    all_y.append(y)

# 3. 최종 병합
all_x_win = np.concatenate(all_x, axis=0)
all_y_win = np.concatenate(all_y, axis=0)

🚀 Extracting features for Evaluation Dataset...
Feature 추출 완료: (3665771, 9)
Feature 추출 완료: (3838860, 9)
Feature 추출 완료: (4443142, 9)
Feature 추출 완료: (4621702, 9)


In [9]:
# ==========================================
# 7. Save
# ==========================================
import numpy as np
np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_0209_511.npz",
    X = all_x_win.astype(np.float32),
    y = all_y_win.astype(np.int64)
    )

print(f" Saved dataset")

print("X shape:", all_x_win.shape)
print("y shape:", all_y_win.shape)

 Saved dataset
X shape: (258893, 128, 9)
y shape: (258893, 128)
